In [19]:
import numpy as np 
import pandas as pd 
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Data Scrap

## yfinance approach: init db, insert data, update db, add new 

In [7]:
import sqlite3
from datetime import datetime, timedelta
import yfinance as yf


In [11]:
DB_NAME = 'stocks.db'


def initialize_database():
    """
    Create a SQLite database and the stock_data table.
    """

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS stock_data (
            stock TEXT,
            date TEXT,
            open REAL,
            high REAL,
            low REAL,
            close REAL,
            volume INTEGER,
            dividends REAL,
            stock_splits REAL,
            PRIMARY KEY (stock, date)
        )
    """)
    conn.commit()
    conn.close()


def insert_data(ticker_symbol, data):
    """
    Insert historical data for a given stock into the database.
    """

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    for _, row in data.iterrows():
        cursor.execute("""
            INSERT OR IGNORE INTO stock_data 
            (stock, date, open, high, low, close, volume, dividends, stock_splits)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, (
            ticker_symbol,
            row['Date'].strftime('%Y-%m-%d') if isinstance(row['Date'], (datetime, pd.Timestamp)) else row['Date'],
            row['Open'],
            row['High'],
            row['Low'],
            row['Close'],
            row['Volume'],
            row['Dividends'],
            row['Stock Splits']
        ))
    conn.commit()
    conn.close()


def fetch_missing_data(ticker_symbol, start_date):
    """
    Fetch missing data for a stock from a given start date.
    """

    ticker = yf.Ticker(ticker_symbol)
    data = ticker.history(start=start_date)
    flat_data = data.reset_index()
    flat_data.insert(0, 'stock', ticker_symbol)
    return flat_data


def update_database():
    """
    Check if stock data is up to date and fetch missing data if needed.
    """

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("SELECT DISTINCT stock FROM stock_data")
    stocks = cursor.fetchall()
    stocks = [stock[0] for stock in stocks]
    
    for stock in stocks:
        cursor.execute("""
            SELECT MAX(date) FROM stock_data WHERE stock = ?
        """, (stock,))
        result = cursor.fetchone()
        last_date = result[0]
        
        if last_date:
            last_date = datetime.strptime(last_date, '%Y-%m-%d').date()
            if last_date < datetime.now().date() - timedelta(days=1):
                print(f"Updating data for {stock}...")
                missing_start = last_date + timedelta(days=1)
                new_data = fetch_missing_data(stock, missing_start)
                insert_data(stock, new_data)
    
    conn.close()


def add_new_stock(ticker_symbol):
    """
    Add a new stock's data to the database.
    """

    print(f"Fetching data for {ticker_symbol}...")
    ticker = yf.Ticker(ticker_symbol)
    historical_data = ticker.history(period='max')
    flat_data = historical_data.reset_index()
    flat_data.insert(0, 'stock', ticker_symbol)
    insert_data(ticker_symbol, flat_data)


def get_data_from_db(ticker_symbol, start_date, end_date):
    """
    Fetch data for a given ticker symbol and date range from the database.

    Parameters:
        ticker_symbol (str): The stock symbol to fetch data for.
        start_date (str): The start date in 'YYYY-MM-DD' format.
        end_date (str): The end date in 'YYYY-MM-DD' format.
    """

    conn = sqlite3.connect(DB_NAME)
    query = """
        SELECT * FROM stock_data
        WHERE stock = ? AND date BETWEEN ? AND ?
        ORDER BY date ASC
    """
    data = pd.read_sql_query(query, conn, params=(ticker_symbol, start_date, end_date))
    conn.close()

    # data['date'] = pd.to_datetime(data['date'])
    return data

In [17]:
# initialize_database()
# add_new_stock('NVDA')
update_database()

Updating data for AAPL...
Updating data for MSFT...
Updating data for NVDA...


# Models

## Download input from DB 

In [15]:
df_test = get_data_from_db('NVDA', '2023-01-01', '2024-12-26')

In [16]:
df_test

,stock,date,open,high,low,close,volume,dividends,stock_splits
0,NVDA,2023-01-03,14.840206,14.985101,14.085754,14.304595,401277000,0.0,0.0
1,NVDA,2023-01-04,14.556414,14.842206,14.230651,14.738281,431324000,0.0,0.0
2,NVDA,2023-01-05,14.480468,14.553414,14.137716,14.254632,389168000,0.0,0.0
3,NVDA,2023-01-06,14.463479,14.999090,14.023800,14.848200,405044000,0.0,0.0
4,NVDA,2023-01-09,15.272891,16.044329,15.129995,15.616641,504231000,0.0,0.0
...,...,...,...,...,...,...,...,...,...
494,NVDA,2024-12-19,131.759995,134.029999,129.550003,130.679993,209719200,0.0,0.0
495,NVDA,2024-12-20,129.809998,135.279999,128.220001,134.699997,306528600,0.0,0.0
496,NVDA,2024-12-23,136.279999,139.789993,135.119995,139.669998,176053500,0.0,0.0
497,NVDA,2024-12-24,140.000000,141.899994,138.649994,140.220001,105157000,0.0,0.0


In [25]:
def split_data(data, train_ratio=0.8):
    """
    Split data into training and validation sets using train_test_split.

    Parameters:
        data (DataFrame): The stock data.
        train_ratio (float): The proportion of data to use for training.
    """

    train_data, validation_data = train_test_split(data, train_size=train_ratio, shuffle=False)
    return train_data, validation_data


def normalize_data(train_data, validation_data):
    """
    Normalize training and validation sets using MinMaxScaler.

    Parameters:
        train_data (DataFrame): Training data.
        validation_data (DataFrame): Validation data.
    """
    columns_to_normalize = ['open', 'high', 'low', 'close']

    scaler = MinMaxScaler(feature_range=(0, 1))
    
    scaler.fit(train_data[columns_to_normalize])
    
    train_data_normalized = train_data.copy()
    validation_data_normalized = validation_data.copy()
    
    train_data_normalized[columns_to_normalize] = scaler.transform(train_data[columns_to_normalize])
    validation_data_normalized[columns_to_normalize] = scaler.transform(validation_data[columns_to_normalize])
    
    
    return train_data_normalized, validation_data_normalized, scaler

In [26]:
train_df_test, validation_df_test = split_data(df_test)


In [27]:
train_df_test

,stock,date,open,high,low,close,volume,dividends,stock_splits
0,NVDA,2023-01-03,14.840206,14.985101,14.085754,14.304595,401277000,0.0,0.0
1,NVDA,2023-01-04,14.556414,14.842206,14.230651,14.738281,431324000,0.0,0.0
2,NVDA,2023-01-05,14.480468,14.553414,14.137716,14.254632,389168000,0.0,0.0
3,NVDA,2023-01-06,14.463479,14.999090,14.023800,14.848200,405044000,0.0,0.0
4,NVDA,2023-01-09,15.272891,16.044329,15.129995,15.616641,504231000,0.0,0.0
...,...,...,...,...,...,...,...,...,...
394,NVDA,2024-07-30,111.502772,111.972700,102.524163,103.713982,486833300,0.0,0.0
395,NVDA,2024-07-31,112.882564,118.321718,110.862872,117.001923,473174200,0.0,0.0
396,NVDA,2024-08-01,117.511853,120.141452,106.793507,109.193138,523462300,0.0,0.0
397,NVDA,2024-08-02,103.743973,108.703206,101.354343,107.253426,482027500,0.0,0.0


In [28]:
validation_df_test

,stock,date,open,high,low,close,volume,dividends,stock_splits
399,NVDA,2024-08-06,103.823962,107.693367,100.534476,104.233902,409012100,0.0,0.0
400,NVDA,2024-08-07,107.793349,108.783202,98.674762,98.894730,411440400,0.0,0.0
401,NVDA,2024-08-08,101.984246,105.483706,97.504935,104.953789,391910000,0.0,0.0
402,NVDA,2024-08-09,105.623688,106.583539,103.414030,104.733826,290844200,0.0,0.0
403,NVDA,2024-08-12,106.303579,111.052845,106.243590,109.003159,325559900,0.0,0.0
...,...,...,...,...,...,...,...,...,...
494,NVDA,2024-12-19,131.759995,134.029999,129.550003,130.679993,209719200,0.0,0.0
495,NVDA,2024-12-20,129.809998,135.279999,128.220001,134.699997,306528600,0.0,0.0
496,NVDA,2024-12-23,136.279999,139.789993,135.119995,139.669998,176053500,0.0,0.0
497,NVDA,2024-12-24,140.000000,141.899994,138.649994,140.220001,105157000,0.0,0.0


In [29]:
train_df_test_scaled, validation_df_test_scaled, scaler = normalize_data(train_df_test, validation_df_test)

In [30]:
train_df_test_scaled

,stock,date,open,high,low,close,volume,dividends,stock_splits
0,NVDA,2023-01-03,0.003006,0.003421,0.000523,0.000412,401277000,0.0,0.0
1,NVDA,2023-01-04,0.000742,0.002289,0.001747,0.003987,431324000,0.0,0.0
2,NVDA,2023-01-05,0.000136,0.000000,0.000962,0.000000,389168000,0.0,0.0
3,NVDA,2023-01-06,0.000000,0.003532,0.000000,0.004893,405044000,0.0,0.0
4,NVDA,2023-01-09,0.006459,0.011815,0.009345,0.011228,504231000,0.0,0.0
...,...,...,...,...,...,...,...,...,...
394,NVDA,2024-07-30,0.774363,0.772036,0.747622,0.737478,486833300,0.0,0.0
395,NVDA,2024-07-31,0.785374,0.822352,0.818065,0.847020,473174200,0.0,0.0
396,NVDA,2024-08-01,0.822315,0.836773,0.783688,0.782647,523462300,0.0,0.0
397,NVDA,2024-08-02,0.712449,0.746126,0.737740,0.766656,482027500,0.0,0.0


In [31]:
validation_df_test_scaled

,stock,date,open,high,low,close,volume,dividends,stock_splits
399,NVDA,2024-08-06,0.713087,0.738123,0.730814,0.741764,409012100,0.0,0.0
400,NVDA,2024-08-07,0.744763,0.746760,0.715104,0.697749,411440400,0.0,0.0
401,NVDA,2024-08-08,0.698407,0.720612,0.705222,0.747699,391910000,0.0,0.0
402,NVDA,2024-08-09,0.727449,0.729328,0.755140,0.745885,290844200,0.0,0.0
403,NVDA,2024-08-12,0.732874,0.764747,0.779043,0.781080,325559900,0.0,0.0
...,...,...,...,...,...,...,...,...,...
494,NVDA,2024-12-19,0.936014,0.946838,0.975928,0.959778,209719200,0.0,0.0
495,NVDA,2024-12-20,0.920453,0.956744,0.964692,0.992918,306528600,0.0,0.0
496,NVDA,2024-12-23,0.972083,0.992485,1.022981,1.033889,176053500,0.0,0.0
497,NVDA,2024-12-24,1.001768,1.009207,1.052802,1.038423,105157000,0.0,0.0


# Tests

In [ ]:
def print_top_records():
    """
    Fetch and print the top 10 records from the stock_data table.
    """
    
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    query = "SELECT * FROM stock_data LIMIT 100;"
    cursor.execute(query)  

    records = cursor.fetchall()
    column_names = [description[0] for description in cursor.description]
    df = pd.DataFrame(records, columns=column_names)
    print(df)
    
    conn.close()

print_top_records()

   stock        date      open      high       low     close      volume  \
0   MSFT  1986-03-13  0.054485  0.062498  0.054485  0.059827  1031788800   
1   MSFT  1986-03-14  0.059827  0.063032  0.059827  0.061963   308160000   
2   MSFT  1986-03-17  0.061963  0.063566  0.061963  0.063032   133171200   
3   MSFT  1986-03-18  0.063032  0.063566  0.060895  0.061429    67766400   
4   MSFT  1986-03-19  0.061429  0.061963  0.059827  0.060361    47894400   
..   ...         ...       ...       ...       ...       ...         ...   
95  MSFT  1986-07-29  0.065169  0.065703  0.062498  0.063566    14054400   
96  MSFT  1986-07-30  0.063566  0.063566  0.059293  0.061429    26409600   
97  MSFT  1986-07-31  0.061429  0.061963  0.060895  0.060895    15638400   
98  MSFT  1986-08-01  0.060895  0.061429  0.059827  0.060361    12902400   
99  MSFT  1986-08-04  0.060361  0.060361  0.058758  0.060361    12441600   

    dividends  stock_splits  
0         0.0           0.0  
1         0.0           0.0